In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.cointegration import compute_returns
from src.cointegration import correlation_matrix
from src.cointegration import generate_candidate_pairs
from src.cointegration import screen_cointegration


# 02 Pair Selection

Generate correlation candidates and apply fixed-orientation Engle–Granger tests. Selection uses raw Engle–Granger p-values at the configured 1% default, without multiple-testing correction. All decisions are saved for inspection.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Settings


In [ ]:
cfg = ResearchConfig().validate()


## 2. Correlation candidates

Use training returns only. Each unordered candidate has one predetermined alphabetical regression direction.


In [ ]:
train_prices = pd.read_parquet("train_prices.parquet")
returns = compute_returns(train_prices)
correlations = correlation_matrix(returns)
candidate_pairs = generate_candidate_pairs(correlations, cfg.correlation_neighbors)
print(f"{len(candidate_pairs):,} candidate pairs from {len(train_prices.columns)} assets")
display(pd.DataFrame(candidate_pairs, columns=["dependent", "independent"]).head(10))


## 3. Cointegration and integration screens

Engle–Granger supplies the cointegration p-value. ADF level/difference screens are diagnostics for the I(1) assumption; they do not prove it.


In [ ]:
selected, spreads, audit = screen_cointegration(
    train_prices,
    candidate_pairs,
    significance=cfg.cointegration_alpha,
    integration_alpha=cfg.integration_alpha,
)
audit.to_parquet("cointegration_audit.parquet")
selected.to_parquet("cointegrated_pairs.parquet")
display(audit[["pair", "pvalue", "multiplicity_method", "integration_screen", "selected"]].head(20))
print(f"{len(selected)} pairs retained")
if selected.empty:
    raise ValueError(
        "No pairs passed. The audit is saved. Review the result before changing thresholds."
    )


## 4. Save fitted spreads

Each column is a selected pair’s formation log-price residual, with alpha and beta recorded in cointegrated_pairs.


In [ ]:
formation_spreads = pd.DataFrame({f"{dep}-{ind}": values for (dep, ind), values in spreads.items()})
formation_spreads.to_parquet("formation_spreads.parquet")
display(selected[["pair", "alpha", "beta", "pvalue"]].head(10))
formation_spreads.iloc[:, :3].plot(figsize=(10, 4), title="Selected formation spreads")
plt.show()
